# Day 4 v2 — Token Analysis: SeanSunny/items_tv_v9

Phân tích độ dài mô tả sản phẩm tiếng Việt để chọn embedding model phù hợp.

**Phân tích:**
- Character count (raw)
- Word count (split by space)
- Token count — `paraphrase-multilingual-MiniLM-L12-v2` (limit=128)
- Token count — `intfloat/multilingual-e5-small` (limit=512)

**Mục tiêu:** Biết chính xác % descriptions bị truncate ở MiniLM (128 tokens) và e5-small (512 tokens).

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

from pricer_vi_2.items import Item

print("Imports OK")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
texts = [item.summary for item in train]
print(f"Train: {len(texts):,} summaries")
print(f"\nSample summary:\n{texts[0]}")

## 2. Character Count

In [ ]:
char_lengths = np.array([len(t) for t in texts])

print("=== Character Count (train 269K) ===")
print(f"Min:   {char_lengths.min():,}")
print(f"Max:   {char_lengths.max():,}")
print(f"Mean:  {char_lengths.mean():.1f}")
print(f"p50:   {np.percentile(char_lengths, 50):.0f}")
print(f"p75:   {np.percentile(char_lengths, 75):.0f}")
print(f"p90:   {np.percentile(char_lengths, 90):.0f}")
print(f"p95:   {np.percentile(char_lengths, 95):.0f}")
print(f"p99:   {np.percentile(char_lengths, 99):.0f}")

## 3. Word Count (split by space)

In [ ]:
word_lengths = np.array([len(t.split()) for t in texts])

print("=== Word Count (split by space) ===")
print(f"Min:   {word_lengths.min():,}")
print(f"Max:   {word_lengths.max():,}")
print(f"Mean:  {word_lengths.mean():.1f}")
print(f"p50:   {np.percentile(word_lengths, 50):.0f}")
print(f"p75:   {np.percentile(word_lengths, 75):.0f}")
print(f"p90:   {np.percentile(word_lengths, 90):.0f}")
print(f"p95:   {np.percentile(word_lengths, 95):.0f}")
print(f"p99:   {np.percentile(word_lengths, 99):.0f}")

## 4. Token Count — MiniLM (limit = 128 tokens)

Model hiện tại trong notebook 03. Đo tỷ lệ bị truncate.

In [ ]:
MINILM_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
MINILM_LIMIT = 128

print(f"Loading tokenizer: {MINILM_NAME}")
minilm_tok = AutoTokenizer.from_pretrained(MINILM_NAME)

print(f"Tokenizing {len(texts):,} texts (no truncation)...")
minilm_lengths = []
for i in range(0, len(texts), 2000):
    batch = texts[i : i + 2000]
    encoded = minilm_tok(batch, truncation=False, padding=False)
    minilm_lengths.extend(len(ids) for ids in encoded["input_ids"])
    if (i // 2000) % 20 == 0:
        print(f"  {i:,}/{len(texts):,}")

minilm_lengths = np.array(minilm_lengths)
pct_over_minilm = (minilm_lengths > MINILM_LIMIT).mean() * 100

print(f"\n=== Token Count — MiniLM (limit={MINILM_LIMIT}) ===")
print(f"Min:   {minilm_lengths.min():,}")
print(f"Max:   {minilm_lengths.max():,}")
print(f"Mean:  {minilm_lengths.mean():.1f}")
print(f"p50:   {np.percentile(minilm_lengths, 50):.0f}")
print(f"p75:   {np.percentile(minilm_lengths, 75):.0f}")
print(f"p90:   {np.percentile(minilm_lengths, 90):.0f}")
print(f"p95:   {np.percentile(minilm_lengths, 95):.0f}")
print(f"p99:   {np.percentile(minilm_lengths, 99):.0f}")
print(f"\n% TRUNCATED at {MINILM_LIMIT} tokens: {pct_over_minilm:.1f}%")

## 5. Token Count — multilingual-e5-small (limit = 512 tokens)

Model thay thế. Cùng dim (384) nhưng limit gấp 4 lần.

In [ ]:
E5_NAME = "intfloat/multilingual-e5-small"
E5_LIMIT = 512

print(f"Loading tokenizer: {E5_NAME}")
e5_tok = AutoTokenizer.from_pretrained(E5_NAME)

print(f"Tokenizing {len(texts):,} texts (no truncation)...")
e5_lengths = []
for i in range(0, len(texts), 2000):
    batch = texts[i : i + 2000]
    encoded = e5_tok(batch, truncation=False, padding=False)
    e5_lengths.extend(len(ids) for ids in encoded["input_ids"])
    if (i // 2000) % 20 == 0:
        print(f"  {i:,}/{len(texts):,}")

e5_lengths = np.array(e5_lengths)
pct_over_e5 = (e5_lengths > E5_LIMIT).mean() * 100

print(f"\n=== Token Count — e5-small (limit={E5_LIMIT}) ===")
print(f"Min:   {e5_lengths.min():,}")
print(f"Max:   {e5_lengths.max():,}")
print(f"Mean:  {e5_lengths.mean():.1f}")
print(f"p50:   {np.percentile(e5_lengths, 50):.0f}")
print(f"p75:   {np.percentile(e5_lengths, 75):.0f}")
print(f"p90:   {np.percentile(e5_lengths, 90):.0f}")
print(f"p95:   {np.percentile(e5_lengths, 95):.0f}")
print(f"p99:   {np.percentile(e5_lengths, 99):.0f}")
print(f"\n% TRUNCATED at {E5_LIMIT} tokens: {pct_over_e5:.1f}%")

## 6. Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(char_lengths, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Character Count Distribution")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")

axes[1].hist(minilm_lengths, bins=50, color="orange", edgecolor="white")
axes[1].axvline(MINILM_LIMIT, color="red", linestyle="--", linewidth=2, label=f"Limit={MINILM_LIMIT}")
axes[1].legend()
axes[1].set_title(f"MiniLM Token Distribution\n({pct_over_minilm:.1f}% truncated)")
axes[1].set_xlabel("Tokens")

axes[2].hist(e5_lengths, bins=50, color="seagreen", edgecolor="white")
axes[2].axvline(E5_LIMIT, color="red", linestyle="--", linewidth=2, label=f"Limit={E5_LIMIT}")
axes[2].legend()
axes[2].set_title(f"e5-small Token Distribution\n({pct_over_e5:.1f}% truncated)")
axes[2].set_xlabel("Tokens")

plt.tight_layout()
plt.savefig("token_analysis.png", dpi=100)
plt.show()
print("Saved token_analysis.png")

## 7. Summary Table

In [ ]:
stats = {
    "min":  lambda a: int(a.min()),
    "max":  lambda a: int(a.max()),
    "mean": lambda a: f"{a.mean():.1f}",
    "p50":  lambda a: int(np.percentile(a, 50)),
    "p90":  lambda a: int(np.percentile(a, 90)),
    "p95":  lambda a: int(np.percentile(a, 95)),
    "p99":  lambda a: int(np.percentile(a, 99)),
}

header = f"{'Metric':<8} {'Chars':>8} {'Words':>8} {'MiniLM':>8} {'e5-small':>10}"
print(header)
print("-" * len(header))
for name, fn in stats.items():
    print(f"{name:<8} {str(fn(char_lengths)):>8} {str(fn(word_lengths)):>8} "
          f"{str(fn(minilm_lengths)):>8} {str(fn(e5_lengths)):>10}")

print()
print(f"% truncated — MiniLM  (@{MINILM_LIMIT} tokens): {pct_over_minilm:.2f}%")
print(f"% truncated — e5-small (@{E5_LIMIT} tokens): {pct_over_e5:.2f}%")
print()
print("=> Kết luận: copy và gửi kết quả này vào chat để chọn model.")